# Datathon 2026 — Faz 2.2: e5-base embedding meta-feature (tek hücre, self-contained)

Bu notebook **src/ import zinciri olmadan** tek hücrede çalışır. İçerik:
- Veri yükleme (train + **test_x**), `/kaggle/input` otomatik bul
- Faz 1/2.1 ile **birebir aynı** fold yapısı (yıl×hedef-desil stratify, SEED=42)
- `text_meta` (word(1,2)+char_wb(3,5) TF-IDF, **alpha=5.0**) + `emb_meta` (e5-base, **ayrı** meta-feature) → CatBoost
- **Test-yıl-ağırlıklı OOF** (public proxy) + yıl-bazlı kırılım (**2026 özel**)
- Gömülü **+0.5 net-bırak kuralı** (Faz 2.1 referansı **87.8292**) → **KEEP/DROP** basar

**Model yükleme:** internet AÇIK ise doğrudan iner. KAPALI ise modeli Dataset olarak ekleyip
hücrenin başındaki `E5_PATH`'i o klasöre ayarla (veya `E5_PATH` env'i ver).

**Sana getirmen gereken 3 satır:** `Faz 2.2 ağırlıklı OOF + kazanç`, `VERDICT`, `2026` satırı.

In [ ]:
# ===================== FAZ 2.2 — TEK HÜCRE =====================
import os, glob, numpy as np, pandas as pd
from scipy.sparse import hstack
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from catboost import CatBoostRegressor, Pool

SEED, N_SPLITS = 42, 5
ID, TARGET, TEXT, YEAR = 'student_id', 'career_success_score', 'mentor_feedback_text', 'application_year'

# --- Model yolu: internet-kapalı için E5_PATH'i Dataset klasörüne çevir ---
E5_PATH = os.environ.get('E5_PATH', 'intfloat/multilingual-e5-base')  # ör: '/kaggle/input/multilingual-e5-base/multilingual-e5-base'

# --- Faz 2.1 referansları (bu repodaki sonuçlardan, net-bırak kuralı için gömülü) ---
REF_F1_W = 92.3796   # Faz 1 metinsiz ağırlıklı OOF
REF_F20_W = 88.4862  # Faz 2.0 word-only ağırlıklı OOF
REF_F21_W = 87.8292  # Faz 2.1 word+char ağırlıklı OOF  <-- net-bırak referansı
REF_F21_BY_YEAR = {2019:54.8414,2020:58.1066,2021:63.5654,2022:76.1679,
                   2023:82.0775,2024:77.8990,2025:106.2116,2026:108.6688}
KEEP_THRESHOLD = 0.5
CHARWORD_ALPHA, EMB_RIDGE_ALPHA = 5.0, 8.0

TR_STOPWORDS = ['ve','ile','bir','bu','da','de','için','çok','daha','gibi','ama','ancak',
                'olarak','ya','ise','en','her','o','ki','mi','ne','fazla','olan','üzerinde',
                'konusunda','birlikte','sağlayabilir','olabilir']
WORD_PARAMS = dict(analyzer='word', ngram_range=(1,2), min_df=3, max_df=0.9, max_features=50000, sublinear_tf=True, lowercase=True)
CHAR_PARAMS = dict(analyzer='char_wb', ngram_range=(3,5), min_df=3, max_features=100000, sublinear_tf=True, lowercase=True)
CATBOOST_PARAMS = dict(loss_function='RMSE', eval_metric='RMSE', iterations=3000, learning_rate=0.03,
                       depth=6, l2_leaf_reg=3.0, random_seed=SEED, od_type='Iter', od_wait=200,
                       verbose=False, allow_writing_files=False)

# ---------- Veri ----------
base = [p for p in glob.glob('/kaggle/input/*') if os.path.exists(f'{p}/train.csv')]
if not base:  # yarışma verisi iç içe klasörde olabilir -> recursive ara
    hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    assert hits, "train.csv bulunamadı — Add Input → Competitions → Datathon 2026 ekli mi?"
    base = [os.path.dirname(hits[0])]
base = base[0]
print("base:", base)
train = pd.read_csv(f'{base}/train.csv'); test = pd.read_csv(f'{base}/test_x.csv')
y = train[TARGET].values
def is_str_col(s): return (s.dtype=='object' or pd.api.types.is_string_dtype(s)) and not pd.api.types.is_numeric_dtype(s)
cat_cols = [c for c in train.columns if c not in (ID,TEXT) and is_str_col(train[c])]
num_cols = [c for c in train.columns if c not in (ID,TARGET,TEXT,*cat_cols)]
text_tr, text_te = train[TEXT].fillna(''), test[TEXT].fillna('')
print(f"train {train.shape} | test_x {test.shape} | {len(num_cols)} sayısal + {len(cat_cols)} kategorik")

# ---------- Fold'lar (Faz 1/2.1 ile birebir) ----------
tbin = pd.qcut(y, 10, labels=False, duplicates='drop')
strat = train[YEAR].astype(str) + '_' + pd.Series(tbin).astype(str)
folds = list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(train, strat))

# ---------- Test-yıl-ağırlıkları (public proxy) ----------
tr_prop = train[YEAR].value_counts(normalize=True); te_prop = test[YEAR].value_counts(normalize=True)
w = np.nan_to_num(train[YEAR].map(lambda yr: te_prop.get(yr,0.0)/tr_prop.get(yr,np.nan)).values)
def wmse(yt, p): return float(np.sum(w*(yt-p)**2)/np.sum(w))
def by_year_mse(p):
    return pd.DataFrame({'year':train[YEAR],'e2':(y-p)**2}).groupby('year')['e2'].mean()

# ---------- text_meta: word+char TF-IDF (alpha=5.0), leakage-free ----------
tm_tr, tm_te = np.zeros(len(train)), np.zeros(len(test))
for tr, va in folds:
    wv = TfidfVectorizer(stop_words=TR_STOPWORDS, **WORD_PARAMS); cv = TfidfVectorizer(**CHAR_PARAMS)
    Xtr = hstack([wv.fit_transform(text_tr.iloc[tr]), cv.fit_transform(text_tr.iloc[tr])]).tocsr()
    Xva = hstack([wv.transform(text_tr.iloc[va]), cv.transform(text_tr.iloc[va])]).tocsr()
    Xte = hstack([wv.transform(text_te), cv.transform(text_te)]).tocsr()
    r = Ridge(alpha=CHARWORD_ALPHA, random_state=SEED).fit(Xtr, y[tr])
    tm_tr[va] = r.predict(Xva); tm_te += r.predict(Xte)/len(folds)
tm_tr, tm_te = np.clip(tm_tr,0,100), np.clip(tm_te,0,100)
print("text_meta hazır.")

# ---------- emb_meta: e5-base (ayrı meta-feature), etiketsiz encode + Ridge-OOF ----------
from sentence_transformers import SentenceTransformer
print(f"e5 yükleniyor: {E5_PATH}")
model = SentenceTransformer(E5_PATH)
emb_tr = model.encode(['query: '+t for t in text_tr.tolist()], batch_size=64, show_progress_bar=True, normalize_embeddings=True)
emb_te = model.encode(['query: '+t for t in text_te.tolist()], batch_size=64, show_progress_bar=True, normalize_embeddings=True)
emb_tr, emb_te = np.asarray(emb_tr), np.asarray(emb_te)
em_tr, em_te = np.zeros(len(train)), np.zeros(len(test))
for tr, va in folds:
    r = Ridge(alpha=EMB_RIDGE_ALPHA, random_state=SEED).fit(emb_tr[tr], y[tr])
    em_tr[va] = r.predict(emb_tr[va]); em_te += r.predict(emb_te)/len(folds)
em_tr, em_te = np.clip(em_tr,0,100), np.clip(em_te,0,100)
e_w = wmse(y, em_tr)
print(f"[embedding-only Ridge] düz={np.mean((y-em_tr)**2):.4f} ağırlıklı={e_w:.4f}")
print(by_year_mse(em_tr).round(3).to_string())

# ---------- CatBoost: sayısal+kategorik + text_meta + emb_meta ----------
features = num_cols + cat_cols + ['text_meta','emb_meta']
X = train[num_cols+cat_cols].copy(); Xt = test[num_cols+cat_cols].copy()
for c in cat_cols: X[c]=X[c].astype(str); Xt[c]=Xt[c].astype(str)
X['text_meta']=tm_tr; Xt['text_meta']=tm_te; X['emb_meta']=em_tr; Xt['emb_meta']=em_te
cat_idx = [features.index(c) for c in cat_cols]
oof, test_pred, imp = np.zeros(len(train)), np.zeros(len(test)), np.zeros(len(features))
for tr, va in folds:
    m = CatBoostRegressor(**CATBOOST_PARAMS)
    m.fit(Pool(X.iloc[tr], y[tr], cat_features=cat_idx),
          eval_set=Pool(X.iloc[va], y[va], cat_features=cat_idx), use_best_model=True)
    oof[va] = m.predict(X.iloc[va]); test_pred += m.predict(Xt)/len(folds); imp += m.get_feature_importance()/len(folds)
oof, test_pred = np.clip(oof,0,100), np.clip(test_pred,0,100)

# ---------- Sonuç + net-bırak verdict ----------
c_w = wmse(y, oof); c_plain = float(np.mean((y-oof)**2))
gain = REF_F21_W - c_w
verdict = 'KEEP' if gain >= KEEP_THRESHOLD else 'DROP'
cby = by_year_mse(oof)
print("\n================ FAZ 2.2 SONUÇ (ağırlıklı OOF) ================")
print(f"Faz 1 (metinsiz)        : {REF_F1_W:.4f}")
print(f"Faz 2.0 (word)          : {REF_F20_W:.4f}")
print(f"Faz 2.1 (word+char)     : {REF_F21_W:.4f}")
print(f"Faz 2.2 (+e5 emb_meta)  : {c_w:.4f}   (düz OOF {c_plain:.4f})")
print(f"  >>> kazanç vs Faz 2.1 : {gain:+.4f}  (eşik {KEEP_THRESHOLD})")
print(f"  >>> emb_meta importance: {pd.Series(imp, index=features)['emb_meta']:.3f}")
print(f"\n>>> NET-BIRAK VERDICT: {verdict} "
      f"({'embedding TUTULUR' if verdict=='KEEP' else 'embedding BIRAKILIR, TF-IDF+char ile devam'})")
print("\nYıl-bazlı kazanç vs Faz 2.1 (pozitif = embedding ek iyileşme):")
for yr in sorted(cby.index):
    d = REF_F21_BY_YEAR.get(int(yr), np.nan) - cby[yr]
    flag = "   <<< 2026: transfer-learning beklentisi" if yr==2026 else ("  <- ağır test payı" if yr==2025 else "")
    print(f"  {yr}: {d:+.4f}{flag}")

# Aday submission (karar netleşince submit)
pd.DataFrame({ID: test[ID].values, TARGET: test_pred}).to_csv('/kaggle/working/sub_faz2_2_emb.csv', index=False)
print("\n[kayıt] /kaggle/working/sub_faz2_2_emb.csv (aday; VERDICT'e göre değerlendir)")
